# 7.2 Transfer Learning
After learning how to combine outputs of multiple different models for the same task, we are now learning how to exploit knowledge from a pretrained model to solve a related but different task, i.e., transfer learning. Specifically, **we apply transfer learning to classify the 10 objects of the CIFAR-10 dataset: airplanes, automobiles, birds, cats, deers, dogs, frogs, horses, ships, and trucks.**

As we will train various CNN architectures, including a pretrained ResNet50 model from Keras,  This allows faster convergence and improved performance with fewer training resources.

This notebook discusses transfer learning using TensorFlow/Keras, the following computations can be expensive. Therefore, we recommend running this notebook on [Google Colab](https://colab.research.google.com). You can upload it to your own Google Drive and open it with Colab. If you don't want to perform the training yourself, you can also directly load the trained models from the folder `trained_models`.


In [ ]:
%pip install tensorflow keras
%pip install upgrade certifi
import matplotlib.pyplot as plt
import os
import pandas as pd
import json
import keras

We start by defining three helper functions:
-  ``_process_history`` converts a training history to a DataFrame and plots training/validation loss and accuracy over epochs
- `_save_trained_model` ensures the `trained_models` directory exists, and then saves the training history as JSON and the Keras model to a .keras file using the provided model_name
- `load_trained_model` loads the Keras model (if it exists) and returns the training history

In [ ]:
def _process_history(history):
    result_frame = pd.DataFrame(history)
    fig, ax = plt.subplots(2, 1, sharex=True, figsize=(10, 8))

    ax[0].plot(
        result_frame[["loss", "val_loss"]],
        label=["Train Loss", "Validation Loss"],
        linewidth=2,
        marker="o",
        markersize=3,
    )

    ax[0].set_ylabel("Loss")
    ax[0].legend(loc="upper left")

    ax[1].plot(
        result_frame[["accuracy", "val_accuracy"]],
        label=["Train Accuracy", "Validation Accuracy"],
        linewidth=2,
        marker="o",
        markersize=3,
    )
    ax[1].set_ylabel("Accuracy")
    ax[1].set_xlabel("Epoch")
    ax[1].legend(loc="upper left")

    plt.subplots_adjust(hspace=0.05)

    plt.show()

def _save_trained_model(model,history, model_name):
    # check if directory exists
    if not os.path.exists(os.path.join(os.getcwd(),"trained_models")):
        os.makedirs(os.path.join(os.getcwd(),"trained_models"))
    with open(os.path.join(os.getcwd(),"trained_models", f"{model_name}_history.json"), 'w') as fp:
        json.dump(history.history, fp)
        
    model.save(os.path.join(os.getcwd(),"trained_models", f"{model_name}.keras"))
    

def _load_trained_model(model_name):
    with open(os.path.join(os.getcwd(),"trained_models", f"{model_name}_history.json"), 'r') as fp:
        history = json.load(fp)
    
    if not os.path.exists(os.path.join(os.getcwd(),"trained_models", f"{model_name}.keras")):
        model = None
    else:
        model = keras.models.load_model(os.path.join(os.getcwd(),"trained_models", f"{model_name}.keras"))
    
    return history, model 

## Data Loading and Preprocessing

We load the CIFAR-10 dataset using TensorFlow/Keras. As preprocessing, we normalize the images to have pixel values between 0 and 1 and convert the labels to categorical encoding.

If you receive this error `URL fetch failure on https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz: None -- [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1006)`, try the following:
- Mac: Move to Applications/Python 3.12 and run `Install Certficates.command`. Restart the kernel and try again.
- Windows: Run this cell `%pip install --upgrade certifi`. Restart the kernel and try again.
- Last resort: Manually download the data from https://www.cs.toronto.edu/~kriz/cifar.html and load it into the notebook.


In [ ]:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.layers import (BatchNormalization, Conv2D,
                                     Dense, Dropout, Flatten, MaxPool2D)
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

# ---- YOUR CODE GOES HERE (START) ---
# load train and test data
(x_train, y_train), (x_test, y_test)=cifar10.load_data()
# normalize pixel values to be between 0 and 1
x_train=x_train/255
x_test=x_test/255
# convert labels to categorical encoding
y_train_cat=to_categorical(y_train,10)
y_test_cat=to_categorical(y_test,10)
# --- YOUR CODE GOES HERE (END) ---

In [ ]:
import matplotlib.pyplot as plt
# define class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
# plot first 25 images from training set with labels
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i])
    plt.xlabel(class_names[y_train[i][0]])
plt.show()

## Base Model

### **TASK: Base Model Training**
We begin with a simple CNN trained directly on the CIFAR-10 dataset which serves as a baseline to understand model behavior without any prior knowledge.

Create a simple CNN model from scratch and train it on the Cifar 10 dataset. Use the keras Sequential API to build the model with several convolutional, pooling, and dense layers. Compile the model with an appropriate optimizer, loss function, and metrics. Train the model for a specified number of epochs and batch size, and evaluate its performance on the test set. Also, save the training history for later analysis. 

Create the following architecture by replacing the ``None`` placeholders with the correct entries:
- Conv2D with 32 filters, kernel size (3,3), activation 'relu', input shape (32, 32, 3)
- MaxPooling2D with pool size (2,2)
- Conv2D with 32 filters, kernel size (3,3), activation 'relu'
- MaxPooling2D with pool size (2,2)
- Flatten
- Dense with 256 units, activation 'relu'
- Dense with 10 units, activation 'softmax'

In [ ]:
BATCH_SIZE = 256
EPOCHS = 100

base_model = Sequential()
# ---- YOUR CODE GOES HERE (START) ---
base_model.add(
    None
)
base_model.add(MaxPool2D(None))
base_model.add(
    Conv2D(None)
)
base_model.add(None)
base_model.add(Flatten())
base_model.add(Dense(None, activation="relu"))
base_model.add(None)
base_model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

base_model_history = base_model.fit(
    None,
    None,
    epochs=None,
    validation_data=(None, y_test_cat),
    batch_size=None,
    verbose=1
)
# --- YOUR CODE GOES HERE (END) ---

_save_trained_model(base_model, base_model_history, "base_model")

### **TASK: Training Curve**
Plot the training curve using the helper function from above and interpret the curve: Was the training successful? Did it stop at the right time?

In [ ]:
# load trained model and history
base_model_history, base_model = _load_trained_model("base_model")
# plot training curve
_process_history(base_model_history)

Interpretation: ...

## Advanced Model

### TASK: Advanced Model Training
In this task, you will create and train a more advanced model including batch normalization, dropout layers, image augmentation and more convolutional layers.

Create the following model by replacing the ``None`` placeholders with the correct entries:
- Each Conv2D-Layer should now use the kernel_initializer he_uniform and should be followed by a BatchNormalization layer. 
- After each MaxPooling2D layer, add a Dropout layer with an increasing drop rate of first 0.2, second 0.3 and 0.4. 
- Also have a 0.5 Dropout before the final Dense layers.
- Use the ImageDataGenerator from keras to apply random transformations to the training images, such as rotations, shifts, and flips, to augment the dataset and improve model generalization. 
- Train the model for a specified number of epochs and batch size, and evaluate its performance on the test set. Also, save the training history for later analysis. The architecture should have the following layers:

In [ ]:
BATCH_SIZE = 256
EPOCHS = 100

advanced_model = Sequential()
# ---- YOUR CODE GOES HERE (START) ---
advanced_model.add(
    Conv2D(
        None
    )
)
advanced_model.add(None)
advanced_model.add(
    None
)
advanced_model.add(None)
advanced_model.add(MaxPool2D(None))
advanced_model.add(None)
advanced_model.add(
    Conv2D(
        None
    )
)
advanced_model.add(None)
advanced_model.add(
    Conv2D(
        None
    )
)
advanced_model.add(None)
advanced_model.add(MaxPool2D(None))
advanced_model.add(Dropout(None))
advanced_model.add(
    Conv2D(
        None
    )
)
advanced_model.add(None)
advanced_model.add(
    Conv2D(
        None
    )
)
advanced_model.add(None)
advanced_model.add(None)
advanced_model.add(Dropout(0.4))
advanced_model.add(None)
advanced_model.add(Dense(None, activation="relu", kernel_initializer=None))
advanced_model.add(None)
advanced_model.add(Dropout(None))
advanced_model.add(Dense(10, activation=None))

advanced_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

datagen = ImageDataGenerator(
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    rotation_range=20,
)
it_train = datagen.flow(None, None)
steps = int(x_train.shape[0] / None)

advanced_model_history = advanced_model.fit(
    it_train,
    epochs=EPOCHS,
    steps_per_epoch=None,
    validation_data=(x_test, y_test_cat),
    batch_size=None,
)
# --- YOUR CODE GOES HERE (END) ---
_save_trained_model(advanced_model, advanced_model_history, "advanced_model")

### **TASK: Training Curve**
Plot the training curve using the helper function from above and interpret the curve: Was the training successful? Did it stop at the right time?

In [ ]:
# load trained model and history
advanced_model_history, advanced_model = _load_trained_model("advanced_model")
# plot training curve
_process_history(advanced_model_history)

Interpretation: ...

## Transfer Learning Model

### **TASK: Transfer Learning with Pretrained ResNet50**
Finally, we apply transfer learning by leveraging the pretrained ResNet50 model. We adapt it to CIFAR-10 by resizing inputs, adding global pooling, and customizing the output layer. This shows how pretrained models can quickly achieve high accuracy.

Create the following model by replacing the ``None`` placeholders with the correct entries:
- Load the ResNet50 model without the top classification layer, using weights pretrained on ImageNet. 
- Instanciate a new Sequential model
- To match the input shape to the Cifar 10 images (32, 32, 3) add an UpSampling2D layer
- Add the ResNet50 base model as the first layer
- Add a GlobalAveragePooling2D layer 
- Add a Dense layer with 10 units and softmax activation for Cifar 10 classification
- Train using the previously defined ImageDataGenerator for data augmentation.


In [ ]:
from keras.applications import ResNet50
from keras import layers

BATCH_SIZE = 64
EPOCHS = 20

# ---- YOUR CODE GOES HERE (START) ---
res_net_base_model=ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

transfer_model = Sequential()
transfer_model.add(None)
transfer_model.add(None)
transfer_model.add(None)
transfer_model.add(None)
# --- YOUR CODE GOES HERE (END) ---

transfer_model.summary()

datagen = ImageDataGenerator(
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    rotation_range=20,
)
it_train = datagen.flow(x_train, y_train_cat)
steps = int(x_train.shape[0] / BATCH_SIZE)

transfer_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

transfer_model_history = transfer_model.fit(
    it_train,
    epochs=EPOCHS,
    steps_per_epoch=steps,
    validation_data=(x_test, y_test_cat),
    batch_size=BATCH_SIZE,
)

_save_trained_model(transfer_model, transfer_model_history, "transfer_model")

### **TASK: Training Curve**
Plot the training curve using the helper function from above and interpret the curve: Was the training successful? Did it stop at the right time?

In [ ]:
# load trained model and history
transfer_model_history, transfer_model = _load_trained_model("transfer_model")
# plot training curve
_process_history(transfer_model_history)

Interpretation: ...

## Performance Comparison

### **TASK: Performance Comparison**
Finally, we can compare the three models (base, advanced, and transfer learning) to analyze how complexity and prior knowledge affect training dynamics and performance. Plot the training and validation accuracy and loss curves for each model using Matplotlib. Analyze the results and discuss the differences in performance. 

Replace tne ``None`` placeholder with the plotting logic.

In [ ]:
def _compare_models(model_histories):
    """
    Compare multiple model training histories.
    
    Parameters:
    -----------
    model_histories : dict
        Dictionary with model names as keys and history dicts as values.
        Example: {
            'base_model': {'loss': [...], 'val_loss': [...], 'accuracy': [...], 'val_accuracy': [...]},
            'advanced_model': {...}
        }
    """
    # ---- YOUR CODE GOES HERE (START) ---
    None
    # --- YOUR CODE GOES HERE (END) ---
    

model_histories = {
    'base_model': base_model_history,
    'advanced_model': advanced_model_history,
    'transfer_model': transfer_model_history
}

_compare_models(model_histories)